In [15]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
print(df.columns)
df.head()


Index(['id', 'sender', 'subject', 'body', 'priority', 'triage_label'], dtype='object')


,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [16]:
# Create empty ground-truth columns
df["ideal_intent"] = ""
df["ideal_tone"] = ""


In [17]:
df.loc[0, ["ideal_intent", "ideal_tone"]] = ["notify", "urgent"]
df.loc[1, ["ideal_intent", "ideal_tone"]] = ["ignore", "polite"]
df.loc[2, ["ideal_intent", "ideal_tone"]] = ["respond", "neutral"]


In [18]:
def email_assistant(email_text):
    text = str(email_text).lower()

    if "urgent" in text or "deadline" in text or "submit" in text:
        return "notify", "urgent"
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"
    else:
        return "respond", "neutral"


In [19]:
predictions = []

for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()


,id,predicted_intent,predicted_tone
0,1,respond,neutral
1,2,respond,neutral
2,3,respond,neutral
3,4,respond,neutral
4,5,respond,neutral


In [20]:
# Merge predictions with ground truth
eval_df = df.merge(pred_df, on="id")


In [21]:
def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score


In [22]:
eval_df["score"] = eval_df.apply(evaluate, axis=1)

accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
accuracy


np.float64(0.5)

In [23]:
eval_df.to_csv(
    "../data/milestone2_output_arbind2.csv",
    index=False
)

print("Evaluation output saved successfully")


Evaluation output saved successfully


In [24]:
df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,urgent
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,ignore,polite
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,,
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,,
